<center>
<img src="https://laelgelcpublic.s3.sa-east-1.amazonaws.com/lael_50_years_narrow_white.png.no_years.400px_96dpi.png" width="300" alt="LAEL 50 years logo">
<h3>APPLIED LINGUISTICS GRADUATE PROGRAMME (LAEL)</h3>
</center>
<hr>

# Corpus Linguistics - Study 1 - Phase 3 - Tatiana

This document aims to organise the tweet verbal and visual subcorpora data for Canonical Correlation Analysis (CCA).

## Tweet Verbal Subcorpus

Define the project name.

In [1]:
PROJECT = "cl_st1_ph2_tatiana"

In [2]:
from pathlib import Path

SCORES_FILE = Path(f"../{PROJECT}/docs/tati_oms/tati_scores_only.tsv")

### Load factor scores

In [3]:
import pandas as pd

scores_verbal_df = pd.read_csv(SCORES_FILE, sep="\t")
scores_verbal_df = scores_verbal_df.rename(columns={"file": "file_id"})

scores_verbal_df

,file_id,fac1,fac2,fac3,fac4,fac5,fac6
0,tweet_00000002,-2,1,0,0,0,0
1,tweet_00000003,-3,2,0,0,0,1
2,tweet_00000004,0,0,-1,0,0,0
3,tweet_00000005,0,0,-1,0,0,0
4,tweet_00000006,1,1,0,-1,0,0
...,...,...,...,...,...,...,...
126863,tweet_00163249,0,0,0,0,0,0
126864,tweet_00163250,1,0,0,0,0,0
126865,tweet_00163251,-1,3,1,-1,0,0
126866,tweet_00163252,0,0,0,0,0,3


### Prepare verbal factor-score columns for CCA

Rename the verbal factor-score columns from `fac<n>` to `ver<n>` and keep only the file identifiers, decade metadata, and renamed verbal score columns needed for Canonical Correlation Analysis.

In [4]:
import re

verbal_factor_cols = [col for col in scores_verbal_df.columns if re.fullmatch(r"fac\d+", col)]
verbal_factor_cols = sorted(verbal_factor_cols, key=lambda col: int(col.replace("fac", "")))

verbal_rename_map = {
    col: col.replace("fac", "ver")
    for col in verbal_factor_cols
}

scores_verbal_df = scores_verbal_df.rename(columns=verbal_rename_map)

verbal_cols = [verbal_rename_map[col] for col in verbal_factor_cols]

scores_verbal_df = scores_verbal_df[
    ["file_id"] + verbal_cols
    ]

scores_verbal_df

,file_id,ver1,ver2,ver3,ver4,ver5,ver6
0,tweet_00000002,-2,1,0,0,0,0
1,tweet_00000003,-3,2,0,0,0,1
2,tweet_00000004,0,0,-1,0,0,0
3,tweet_00000005,0,0,-1,0,0,0
4,tweet_00000006,1,1,0,-1,0,0
...,...,...,...,...,...,...,...
126863,tweet_00163249,0,0,0,0,0,0
126864,tweet_00163250,1,0,0,0,0,0
126865,tweet_00163251,-1,3,1,-1,0,0
126866,tweet_00163252,0,0,0,0,0,3


### Load tweets corpus

In [5]:
TWEETS_FILE = Path(f"../{PROJECT}/corpus/tweets.ndjson")

tweets_df = pd.read_json(TWEETS_FILE, lines=True)

tweets_df

,file,created_at,text_uniq,conversation_id,author_id,username,tweet_url,text,photo_uniq_id
0,saude202003_n_00001.json,2020-03-30 23:58:27,tweet_00000006,convo_1244774019859206145,id_173533635,unknown,unknown,"Lembrando q estamos atendendo somente on-line,...",saude202003_n_00001_00000001.jpg
1,saude202003_n_00001.json,2020-03-30 23:54:45,tweet_00000007,convo_1244774019859206145,id_173533635,unknown,unknown,"Não é muito, mas é a forma que encontramos par...",saude202003_n_00001_00000001.jpg
2,saude202003_n_00001.json,2020-03-30 23:50:57,tweet_00000022,convo_1244774142701981698,id_454396343,unknown,unknown,"Segundo levantamento da AFP (dados da OMS) 3,3...",saude202003_n_00001_00000002.jpg
3,saude202003_n_00002.json,2020-03-30 23:27:18,tweet_00000051,convo_1244693362361487364,id_191068627,sf79,https://twitter.com/sf79/status/12446933623614...,@cmelo @Francischini_ @jairbolsonaro A OMS nun...,saude202003_n_00002_00000001.jpg
4,saude202003_n_00002.json,2020-03-30 23:26:49,tweet_00000054,convo_1244768070524166145,id_199501770,unknown,unknown,OMS diz que não há transmissão aérea de Covid-...,saude202003_n_00002_00000002.jpg
...,...,...,...,...,...,...,...,...,...
15782,saude202303_n_00207.json,2023-03-06 23:52:37,tweet_00163212,convo_1632891969536966663,id_1485793803495641090,VascoN95,https://twitter.com/VascoN95/status/1632891969...,Neste guia atualizado em fevereiro de 2023 sob...,saude202303_n_00207_00000001.jpg
15783,saude202303_n_00207.json,2023-03-07 09:36:34,tweet_00163219,convo_1633038922753122305,id_1501277332119511048,chris_banida,https://twitter.com/chris_banida/status/163303...,Em breve esse venen☠️ virá para as crianças. h...,saude202303_n_00207_00000002.jpg
15784,saude202303_n_00207.json,2023-03-07 08:45:13,tweet_00163225,convo_1633026002245890049,id_2370633043,unknown,unknown,Aumenta a preocupação da OMS com a gripe aviár...,saude202303_n_00207_00000003.jpg
15785,saude202303_n_00207.json,2023-03-07 02:30:35,tweet_00163233,convo_1632931722000990210,id_1602447377771773953,anamanerolemos,https://twitter.com/anamanerolemos/status/1632...,A OMS muda seu plano de pandemia (que não menc...,saude202303_n_00207_00000004.png


### Add image IDs to verbal factor scores

In [6]:
image_id_map = tweets_df.set_index("text_uniq")["photo_uniq_id"]

scores_verbal_df.insert(
    loc=scores_verbal_df.columns.get_loc("file_id") + 1,
    column="image_id",
    value=scores_verbal_df["file_id"].map(image_id_map),
)

scores_verbal_df

,file_id,image_id,ver1,ver2,ver3,ver4,ver5,ver6
0,tweet_00000002,NaN,-2,1,0,0,0,0
1,tweet_00000003,NaN,-3,2,0,0,0,1
2,tweet_00000004,NaN,0,0,-1,0,0,0
3,tweet_00000005,NaN,0,0,-1,0,0,0
4,tweet_00000006,saude202003_n_00001_00000001.jpg,1,1,0,-1,0,0
...,...,...,...,...,...,...,...,...
126863,tweet_00163249,NaN,0,0,0,0,0,0
126864,tweet_00163250,saude202303_n_00207_00000005.jpg,1,0,0,0,0,0
126865,tweet_00163251,NaN,-1,3,1,-1,0,0
126866,tweet_00163252,NaN,0,0,0,0,0,3


## Tweet Visual Subcorpus

Define the project name.

In [7]:
PROJECT = "cl_st1_ph2_tatiana"

In [8]:
from pathlib import Path

SCORES_FILE = Path(f"../{PROJECT}/sas/output_{PROJECT}/{PROJECT}_scores_only.tsv")
FILE_IDS_PATH = Path(f"../{PROJECT}/file_ids.txt")

### Load factor scores

In [9]:
import pandas as pd

scores_visual_df = pd.read_csv(SCORES_FILE, sep="\t")
scores_visual_df = scores_visual_df.rename(columns={"filename": "text_id"})

scores_visual_df

,text_id,fac1,fac2,fac3,fac4,fac5,fac6,fac7
0,t000001,0,0,0,0,0,0,-2
1,t000002,1,0,0,1,0,0,-2
2,t000003,6,0,0,0,0,0,-6
3,t000004,1,0,0,1,0,0,-3
4,t000005,19,-1,0,1,0,1,0
...,...,...,...,...,...,...,...,...
11844,t011865,0,0,0,0,0,0,-4
11845,t011866,3,0,0,0,0,0,-4
11846,t011867,0,0,0,0,0,-5,0
11847,t011868,0,8,0,4,0,0,19


### Load file ID mapping

In [10]:
file_ids_visual_df = pd.read_csv(
    FILE_IDS_PATH,
    sep=" ",
    names=["text_id", "filename"],
)

file_ids_visual_df.head()

,text_id,filename
0,t000001,saude202003_n_00001_00000001.txt
1,t000002,saude202003_n_00001_00000002.txt
2,t000003,saude202003_n_00002_00000001.txt
3,t000004,saude202003_n_00002_00000002.txt
4,t000005,saude202003_n_00002_00000003.txt


### Merge factor scores with file ID mapping

In [11]:
scores_file_ids_visual_df = scores_visual_df.merge(file_ids_visual_df, on="text_id", how="left")
scores_file_ids_visual_df = scores_file_ids_visual_df[
    ["text_id", "filename"]
    + [col for col in scores_file_ids_visual_df.columns if col not in ["text_id", "filename"]]
    ]

scores_file_ids_visual_df

,text_id,filename,fac1,fac2,fac3,fac4,fac5,fac6,fac7
0,t000001,saude202003_n_00001_00000001.txt,0,0,0,0,0,0,-2
1,t000002,saude202003_n_00001_00000002.txt,1,0,0,1,0,0,-2
2,t000003,saude202003_n_00002_00000001.txt,6,0,0,0,0,0,-6
3,t000004,saude202003_n_00002_00000002.txt,1,0,0,1,0,0,-3
4,t000005,saude202003_n_00002_00000003.txt,19,-1,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...
11844,t011865,saude202303_n_00207_00000001.txt,0,0,0,0,0,0,-4
11845,t011866,saude202303_n_00207_00000002.txt,3,0,0,0,0,0,-4
11846,t011867,saude202303_n_00207_00000003.txt,0,0,0,0,0,-5,0
11847,t011868,saude202303_n_00207_00000004.txt,0,8,0,4,0,0,19


### Prepare visual factor-score columns for CCA

Rename the visual factor-score columns from `fac<n>` to `vis<n>` and keep only the file identifiers, decade metadata, and renamed visual score columns needed for Canonical Correlation Analysis.

In [12]:
import re

verbal_factor_cols = [col for col in scores_file_ids_visual_df.columns if re.fullmatch(r"fac\d+", col)]
verbal_factor_cols = sorted(verbal_factor_cols, key=lambda col: int(col.replace("fac", "")))

verbal_rename_map = {
    col: col.replace("fac", "vis")
    for col in verbal_factor_cols
}

scores_file_ids_visual_df = scores_file_ids_visual_df.rename(columns=verbal_rename_map)

verbal_cols = [verbal_rename_map[col] for col in verbal_factor_cols]

scores_file_ids_visual_df = scores_file_ids_visual_df[
    ["text_id", "filename"] + verbal_cols
    ]

scores_file_ids_visual_df

,text_id,filename,vis1,vis2,vis3,vis4,vis5,vis6,vis7
0,t000001,saude202003_n_00001_00000001.txt,0,0,0,0,0,0,-2
1,t000002,saude202003_n_00001_00000002.txt,1,0,0,1,0,0,-2
2,t000003,saude202003_n_00002_00000001.txt,6,0,0,0,0,0,-6
3,t000004,saude202003_n_00002_00000002.txt,1,0,0,1,0,0,-3
4,t000005,saude202003_n_00002_00000003.txt,19,-1,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...
11844,t011865,saude202303_n_00207_00000001.txt,0,0,0,0,0,0,-4
11845,t011866,saude202303_n_00207_00000002.txt,3,0,0,0,0,0,-4
11846,t011867,saude202303_n_00207_00000003.txt,0,0,0,0,0,-5,0
11847,t011868,saude202303_n_00207_00000004.txt,0,8,0,4,0,0,19


### Add verbal file IDs to visual scores

In [13]:
visual_to_verbal_file_id_map = (
    scores_verbal_df
    .dropna(subset=["image_id"])
    .assign(image_stem=lambda df: df["image_id"].map(lambda value: Path(value).stem))
    .drop_duplicates(subset="image_stem")
    .set_index("image_stem")["file_id"]
)

visual_filename_stems = scores_file_ids_visual_df["filename"].map(lambda value: Path(value).stem)

scores_file_ids_visual_df.insert(
    loc=scores_file_ids_visual_df.columns.get_loc("text_id") + 1,
    column="file_id",
    value=visual_filename_stems.map(visual_to_verbal_file_id_map),
)

scores_file_ids_visual_df

,text_id,file_id,filename,vis1,vis2,vis3,vis4,vis5,vis6,vis7
0,t000001,tweet_00000006,saude202003_n_00001_00000001.txt,0,0,0,0,0,0,-2
1,t000002,tweet_00000022,saude202003_n_00001_00000002.txt,1,0,0,1,0,0,-2
2,t000003,tweet_00000051,saude202003_n_00002_00000001.txt,6,0,0,0,0,0,-6
3,t000004,tweet_00000054,saude202003_n_00002_00000002.txt,1,0,0,1,0,0,-3
4,t000005,tweet_00000073,saude202003_n_00002_00000003.txt,19,-1,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...
11844,t011865,tweet_00163212,saude202303_n_00207_00000001.txt,0,0,0,0,0,0,-4
11845,t011866,tweet_00163219,saude202303_n_00207_00000002.txt,3,0,0,0,0,0,-4
11846,t011867,tweet_00163225,saude202303_n_00207_00000003.txt,0,0,0,0,0,-5,0
11847,t011868,tweet_00163233,saude202303_n_00207_00000004.txt,0,8,0,4,0,0,19


## Identify misalignment between visual and verbal dimension score DataFrames

In [14]:
verbal_file_ids = set(scores_verbal_df["file_id"])
visual_file_ids = set(scores_file_ids_visual_df["file_id"])

file_id_diff = {
    "only_in_verbal": sorted(verbal_file_ids - visual_file_ids),
    "only_in_visual": sorted(visual_file_ids - verbal_file_ids),
    "n_only_in_verbal": len(verbal_file_ids - visual_file_ids),
    "n_only_in_visual": len(visual_file_ids - verbal_file_ids),
    "n_in_both": len(verbal_file_ids & visual_file_ids),
    "n_verbal": len(verbal_file_ids),
    "n_visual": len(visual_file_ids),
}

display(file_id_diff)

{'only_in_verbal': ['tweet_00000002',
  'tweet_00000003',
  'tweet_00000004',
  'tweet_00000005',
  'tweet_00000007',
  'tweet_00000008',
  'tweet_00000009',
  'tweet_00000010',
  'tweet_00000011',
  'tweet_00000015',
  'tweet_00000016',
  'tweet_00000017',
  'tweet_00000018',
  'tweet_00000019',
  'tweet_00000020',
  'tweet_00000021',
  'tweet_00000023',
  'tweet_00000024',
  'tweet_00000025',
  'tweet_00000026',
  'tweet_00000027',
  'tweet_00000030',
  'tweet_00000031',
  'tweet_00000032',
  'tweet_00000033',
  'tweet_00000034',
  'tweet_00000035',
  'tweet_00000037',
  'tweet_00000038',
  'tweet_00000039',
  'tweet_00000040',
  'tweet_00000041',
  'tweet_00000042',
  'tweet_00000043',
  'tweet_00000044',
  'tweet_00000045',
  'tweet_00000046',
  'tweet_00000047',
  'tweet_00000048',
  'tweet_00000049',
  'tweet_00000050',
  'tweet_00000052',
  'tweet_00000053',
  'tweet_00000055',
  'tweet_00000056',
  'tweet_00000057',
  'tweet_00000058',
  'tweet_00000059',
  'tweet_00000060',
  

## Merge verbal and visual scores for CCA

Create the CCA-ready dataset by inner-merging the verbal and visual score DataFrames on `file_id`, keeping only commercials present in both modalities.

In [15]:
cca_df = scores_verbal_df.merge(
    scores_file_ids_visual_df,
    on="file_id",
    how="inner",
)

cca_df

,file_id,image_id,ver1,ver2,ver3,ver4,ver5,ver6,text_id,filename,vis1,vis2,vis3,vis4,vis5,vis6,vis7
0,tweet_00000006,saude202003_n_00001_00000001.jpg,1,1,0,-1,0,0,t000001,saude202003_n_00001_00000001.txt,0,0,0,0,0,0,-2
1,tweet_00000022,saude202003_n_00001_00000002.jpg,-1,3,0,0,0,0,t000002,saude202003_n_00001_00000002.txt,1,0,0,1,0,0,-2
2,tweet_00000051,saude202003_n_00002_00000001.jpg,-1,2,0,-1,-1,0,t000003,saude202003_n_00002_00000001.txt,6,0,0,0,0,0,-6
3,tweet_00000054,saude202003_n_00002_00000002.jpg,1,1,-1,0,0,0,t000004,saude202003_n_00002_00000002.txt,1,0,0,1,0,0,-3
4,tweet_00000073,saude202003_n_00002_00000003.jpg,1,2,0,-1,0,0,t000005,saude202003_n_00002_00000003.txt,19,-1,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11844,tweet_00163212,saude202303_n_00207_00000001.jpg,0,1,0,1,0,2,t011865,saude202303_n_00207_00000001.txt,0,0,0,0,0,0,-4
11845,tweet_00163219,saude202303_n_00207_00000002.jpg,0,0,0,0,0,1,t011866,saude202303_n_00207_00000002.txt,3,0,0,0,0,0,-4
11846,tweet_00163225,saude202303_n_00207_00000003.jpg,0,2,0,0,0,0,t011867,saude202303_n_00207_00000003.txt,0,0,0,0,0,-5,0
11847,tweet_00163233,saude202303_n_00207_00000004.png,0,3,0,-1,0,0,t011868,saude202303_n_00207_00000004.txt,0,8,0,4,0,0,19


In [16]:
cca_df = cca_df.drop(
    columns=[
        "text_id",
        "filename",
    ]
)

cca_df

,file_id,image_id,ver1,ver2,ver3,ver4,ver5,ver6,vis1,vis2,vis3,vis4,vis5,vis6,vis7
0,tweet_00000006,saude202003_n_00001_00000001.jpg,1,1,0,-1,0,0,0,0,0,0,0,0,-2
1,tweet_00000022,saude202003_n_00001_00000002.jpg,-1,3,0,0,0,0,1,0,0,1,0,0,-2
2,tweet_00000051,saude202003_n_00002_00000001.jpg,-1,2,0,-1,-1,0,6,0,0,0,0,0,-6
3,tweet_00000054,saude202003_n_00002_00000002.jpg,1,1,-1,0,0,0,1,0,0,1,0,0,-3
4,tweet_00000073,saude202003_n_00002_00000003.jpg,1,2,0,-1,0,0,19,-1,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11844,tweet_00163212,saude202303_n_00207_00000001.jpg,0,1,0,1,0,2,0,0,0,0,0,0,-4
11845,tweet_00163219,saude202303_n_00207_00000002.jpg,0,0,0,0,0,1,3,0,0,0,0,0,-4
11846,tweet_00163225,saude202303_n_00207_00000003.jpg,0,2,0,0,0,0,0,0,0,0,0,-5,0
11847,tweet_00163233,saude202303_n_00207_00000004.png,0,3,0,-1,0,0,0,8,0,4,0,0,19


### Save the verbal and visual scores as NDJSON, Excel, and TSV

In [17]:
cca_df.to_json(
    "../cl_st1_ph2_tatiana/corpus/tweets_cca.ndjson",
    orient="records",
    lines=True,
    force_ascii=False
)

cca_df.to_excel(
    "../cl_st1_ph2_tatiana/corpus/tweets_cca.xlsx",
    index=False,
    engine="openpyxl"
)

cca_df.to_csv(
    "../cl_st1_ph2_tatiana/corpus/tweets_cca.tsv",
    sep="\t",
    index=False
)